In [68]:
import tensorflow as tf
import pickle
import pandas as pd
import numpy as np

In [69]:
data = pd.read_csv("Churn_Modelling.csv")

In [70]:
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [71]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [72]:
from sklearn.preprocessing import LabelEncoder
lb = LabelEncoder()
data["Gender"] = lb.fit_transform(data["Gender"])

In [73]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [74]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
geo_encoded = ohe.fit_transform(data[["Geography"]]).toarray()

In [75]:
ohe.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [76]:
geo_encoded_df = pd.DataFrame(data=geo_encoded, columns=ohe.get_feature_names_out())

In [77]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [78]:
data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)

In [79]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [80]:
# Split the data into features and target
X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']

In [82]:
## Split the data in training and tetsing sets
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [84]:
## Scale these features
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

#### ANN Regression Problem statement

In [85]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [87]:
# Build the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression # No need of specifying activation function in regression ps
])

## compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [88]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Set up TensorBoard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [89]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


In [90]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 5ms/step - loss: 100399.0312 - mae: 100399.0312 - val_loss: 98594.1250 - val_mae: 98594.1250
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 99897.3594 - mae: 99897.3594 - val_loss: 97560.5625 - val_mae: 97560.5625
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 98068.7891 - mae: 98068.7891 - val_loss: 94840.5078 - val_mae: 94840.5078
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 94363.6562 - mae: 94363.6562 - val_loss: 90141.2812 - val_mae: 90141.2812
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 88721.4141 - mae: 88721.4141 - val_loss: 83706.8906 - val_mae: 83706.8906
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 81538.8750 - mae: 81538.8750 - val_loss: 76185.7656 - val_mae: 76185.7656
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 73744.4062 - mae: 73744.406

In [91]:
%load_ext tensorboard

In [92]:
%tensorboard --logdir regressionlogs/fit